## TPOT Regressor


## Parameters

In [1]:
# To ignore warinings
import warnings
warnings.filterwarnings('ignore')

In [2]:
NUM_FOLDS = 7
SEED = 42

## TabNet Parameters

In [3]:
## TabNet Parameters
MAX_EPOCH = 300
N_D = 6 
N_A = 6 
N_STEPS = 2
GAMMA = 1.1
LAMBDA_SPARSE = 1e-4
OPT_LR = 1e-4
OPT_WEIGHT_DECAY = 1e-5
OPT_MOMENTUM = 0.9
MASK_TYPE = "entmax"
SCHEDULER_MIN_LR = 1e-6
SCHEDULER_FACTOR = 0.9
DEVICE_NAME = "cuda"

BATCH_SIZE = 32

## Imports Libs

In [4]:
import torch
from torch import nn

from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd 

import os
import random
import sys
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
seed_everything(SEED)

## Import Data

In [5]:
import pandas as pd

train = pd.read_csv("/kaggle/input/zelestra/train.csv")

train = train.drop(columns=['id'])
test=pd.read_csv("/kaggle/input/zelestra/test.csv").drop(columns=['id'])

In [6]:
train.dtypes

temperature           float64
irradiance            float64
humidity               object
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed             object
pressure               object
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [7]:
columns = train.columns.to_list()
features = columns[:-1]
target = columns[-1]

print(f'Features: {features}')
print(f'Target: {target}')

Features: ['temperature', 'irradiance', 'humidity', 'panel_age', 'maintenance_count', 'soiling_ratio', 'voltage', 'current', 'module_temperature', 'cloud_coverage', 'wind_speed', 'pressure', 'string_id', 'error_code', 'installation_type']
Target: efficiency


In [8]:
obj_num_cols=train.select_dtypes(include='object').columns[:-3]

In [9]:
obj_num_cols

Index(['humidity', 'wind_speed', 'pressure'], dtype='object')

In [10]:
for col in obj_num_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')

In [11]:
train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [12]:
for col in obj_num_cols:
    test[col] = pd.to_numeric(test[col], errors='coerce')

In [13]:
X_train = train[features]
y_train = train[target]

In [14]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [15]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')


pt.fit(X_train[list(X_train.select_dtypes(include='float64').columns)])
X_train_pt = pt.transform(X_train[list(X_train.select_dtypes(include='float64').columns)])

print(f'Shape of X_train_pt: {X_train_pt.shape}')

Shape of X_train_pt: (20000, 12)


In [16]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [17]:
pt.fit(test[list(test.select_dtypes(include='float64').columns)])
test_pt = pt.transform(test[list(test.select_dtypes(include='float64').columns)])

In [18]:
test_pt=pd.DataFrame(test_pt,columns=list(test.select_dtypes(include='float64').columns))
X_train_pt = pd.DataFrame(X_train_pt, columns=list(X_train.select_dtypes(include='float64').columns))

In [19]:
X_train_pt[list(X_train.select_dtypes(include='object').columns)]=X_train[list(X_train.select_dtypes(include='object').columns)]

In [20]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [21]:
test_pt[list(test.select_dtypes(include='object').columns)]=test[list(test.select_dtypes(include='object').columns)]

In [22]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [23]:
from sklearn.preprocessing import LabelEncoder

# Sample: list of columns to encode
cols_to_encode = list(X_train_pt.select_dtypes(include='object').columns)

le = LabelEncoder()
for col in cols_to_encode:
    # Convert all values to string to avoid mixed types (also handles NaN as 'nan')
    X_train_pt[col] = X_train_pt[col].astype(str)
    X_train_pt[col] = le.fit_transform(X_train_pt[col])

In [24]:
for col in cols_to_encode:
    test_pt[col] = test_pt[col].astype(str)
    test_pt[col] = le.fit_transform(test_pt[col])

In [25]:
target = train.pop("efficiency")
target = target.values

## Label Encoding

In [26]:
columns = test.columns

## Create TabNet Params Dictionary

In [27]:
import numpy as np

X_train_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
test_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
from sklearn.impute import SimpleImputer

numeric_cols = X_train_pt.select_dtypes(include=[np.number]).columns

# Create the imputer (you can use 'mean', 'median', or 'most_frequent')
imputer = SimpleImputer(strategy='mean')

# Fit and transform only on numeric columns
X_train_pt[numeric_cols] = imputer.fit_transform(X_train_pt[numeric_cols])
numeric_cols = test_pt.select_dtypes(include=[np.number]).columns
test_pt[numeric_cols] = imputer.fit_transform(test_pt[numeric_cols])


In [28]:
list(X_train_pt.columns)[12:]

['string_id', 'error_code', 'installation_type']

In [29]:
column = list(X_train_pt.columns)
column.remove('string_id')
column.remove('error_code')
column.remove('installation_type')

for i in column[6:]:
    for j in column[6:]:
        print(i,j)
        if i != j:
            X_train_pt[f'{i} x {j}'] = (X_train_pt[i].astype('float').to_numpy() * X_train_pt[j].astype('float').to_numpy())
            test_pt[f'{i} x {j}'] = (test_pt[i].astype('float').to_numpy() * test_pt[j].astype('float').to_numpy())

voltage voltage
voltage current
voltage module_temperature
voltage cloud_coverage
voltage wind_speed
voltage pressure
current voltage
current current
current module_temperature
current cloud_coverage
current wind_speed
current pressure
module_temperature voltage
module_temperature current
module_temperature module_temperature
module_temperature cloud_coverage
module_temperature wind_speed
module_temperature pressure
cloud_coverage voltage
cloud_coverage current
cloud_coverage module_temperature
cloud_coverage cloud_coverage
cloud_coverage wind_speed
cloud_coverage pressure
wind_speed voltage
wind_speed current
wind_speed module_temperature
wind_speed cloud_coverage
wind_speed wind_speed
wind_speed pressure
pressure voltage
pressure current
pressure module_temperature
pressure cloud_coverage
pressure wind_speed
pressure pressure


In [30]:
len(X_train_pt.columns)==len(test_pt.columns)

True

In [31]:
y_train

0        0.562096
1        0.396447
2        0.573776
3        0.629009
4        0.341874
           ...   
19995    0.664907
19996    0.354070
19997    0.419734
19998    0.661963
19999    0.714566
Name: efficiency, Length: 20000, dtype: float64

In [32]:
X_train_pt

,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,...,wind_speed x voltage,wind_speed x current,wind_speed x module_temperature,wind_speed x cloud_coverage,wind_speed x pressure,pressure x voltage,pressure x current,pressure x module_temperature,pressure x cloud_coverage,pressure x wind_speed
0,-1.461798e+00,2.981458e-01,-0.209258,1.353260,0.083994,6.150617e-01,1.157421,0.383845,-1.353703,5.009804e-01,...,1.384920,0.459292,-1.619782,5.994516e-01,0.701423,0.678481,0.225010,-0.793542,2.936752e-01,0.701423
1,4.945863e-02,-1.041887e+00,-1.980590,0.315154,1.802125,-1.283641e+00,0.668014,-1.531609,-0.168763,8.913644e-03,...,0.693363,-1.589730,-0.175167,9.251896e-03,1.305876,0.840450,-1.926969,-0.212326,1.121455e-02,1.305876
2,1.616598e+00,7.426669e-01,1.336257,-1.762462,0.083994,7.241862e-01,1.411803,1.775808,1.097742,8.905170e-17,...,-1.884507,-2.370390,-1.465291,-1.188683e-16,0.272800,-0.288533,-0.362925,-0.224347,-1.819966e-17,0.272800
3,2.052147e+00,9.322996e-01,1.473734,0.178125,-0.433087,8.099324e-01,1.369926,-0.560946,2.198234,6.144260e-01,...,0.503990,-0.206370,0.808722,2.260449e-01,0.324706,1.209100,-0.495093,1.940168,5.422941e-01,0.324706
4,-1.707871e+00,-1.947823e+00,-0.712139,1.238835,0.997521,-8.488522e-01,-1.341608,-0.632271,-1.993974,-1.981372e+00,...,2.355281,1.109994,3.500552,3.478429e+00,0.772526,0.590366,0.278227,0.877436,8.718908e-01,0.772526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,-6.031200e-01,3.064456e-16,1.399716,-0.215810,-0.433087,2.470165e-01,0.205567,1.115200,-0.279409,-2.241137e+00,...,0.236771,1.284481,-0.321822,-2.581330e+00,0.618806,0.110442,0.599145,-0.150114,-1.204060e+00,0.618806
19996,2.056977e+00,-8.149563e-01,1.412425,0.845342,-1.010662,-1.080966e+00,-1.341608,-1.094062,2.743025,5.497263e-01,...,2.145471,1.749602,-4.386589,-8.791109e-01,-0.494272,-0.414662,-0.338151,0.847810,1.699087e-01,-0.494272
19997,-2.096698e+00,6.338132e-01,-0.324359,1.408085,2.174826,-8.681263e-01,0.258084,1.716960,-1.543985,3.846898e-01,...,-0.139841,-0.930323,0.836597,-2.084415e-01,0.177541,-0.084564,-0.562581,0.505904,-1.260480e-01,0.177541
19998,2.847930e-16,5.238397e-01,-0.147997,0.231199,0.083994,8.426236e-16,-1.341608,-0.441677,-0.704003,8.488560e-01,...,-1.204179,-0.396434,-0.631888,7.619029e-01,-0.563226,0.841865,0.277154,0.441765,-5.326610e-01,-0.563226


In [33]:
from tpot.config import regressor_config_dict

# Print all available models
for model in regressor_config_dict:
    print(model)


sklearn.linear_model.ElasticNetCV
sklearn.ensemble.ExtraTreesRegressor
sklearn.ensemble.GradientBoostingRegressor
sklearn.ensemble.AdaBoostRegressor
sklearn.tree.DecisionTreeRegressor
sklearn.neighbors.KNeighborsRegressor
sklearn.linear_model.LassoLarsCV
sklearn.svm.LinearSVR
sklearn.ensemble.RandomForestRegressor
sklearn.linear_model.RidgeCV
xgboost.XGBRegressor
sklearn.linear_model.SGDRegressor
sklearn.preprocessing.Binarizer
sklearn.decomposition.FastICA
sklearn.cluster.FeatureAgglomeration
sklearn.preprocessing.MaxAbsScaler
sklearn.preprocessing.MinMaxScaler
sklearn.preprocessing.Normalizer
sklearn.kernel_approximation.Nystroem
sklearn.decomposition.PCA
sklearn.preprocessing.PolynomialFeatures
sklearn.kernel_approximation.RBFSampler
sklearn.preprocessing.RobustScaler
sklearn.preprocessing.StandardScaler
tpot.builtins.ZeroCount
tpot.builtins.OneHotEncoder
sklearn.feature_selection.SelectFwe
sklearn.feature_selection.SelectPercentile
sklearn.feature_selection.VarianceThreshold
sklear

In [34]:
from tpot.config import regressor_config_dict

custom_config = {
    # Regressors
    'sklearn.linear_model.RidgeCV': regressor_config_dict['sklearn.linear_model.RidgeCV'],
    'sklearn.ensemble.RandomForestRegressor': regressor_config_dict['sklearn.ensemble.RandomForestRegressor'],
    'sklearn.ensemble.GradientBoostingRegressor': regressor_config_dict['sklearn.ensemble.GradientBoostingRegressor'],
    'xgboost.XGBRegressor': regressor_config_dict['xgboost.XGBRegressor'],

    # Preprocessing (only scalers)
    'sklearn.preprocessing.StandardScaler': regressor_config_dict['sklearn.preprocessing.StandardScaler'],
    'sklearn.preprocessing.MinMaxScaler': regressor_config_dict['sklearn.preprocessing.MinMaxScaler'],
    'sklearn.preprocessing.RobustScaler': regressor_config_dict['sklearn.preprocessing.RobustScaler']
}


In [35]:
from tpot import TPOTRegressor
from sklearn.metrics import mean_squared_error

tpot = TPOTRegressor(
    generations=7,
    population_size=20,
    verbosity=2,
    scoring='neg_mean_squared_error',
    config_dict=custom_config,
    random_state=42,
    disable_update_check=True
)

tpot.fit(X_train_pt, y_train)


Optimization Progress:   0%|          | 0/160 [00:00<?, ?pipeline/s]


Generation 1 - Current best internal CV score: -0.010771350847619415

Generation 2 - Current best internal CV score: -0.010664221241557784

Generation 3 - Current best internal CV score: -0.010626407922470287

Generation 4 - Current best internal CV score: -0.010626407922470287

Generation 5 - Current best internal CV score: -0.010626407922470287

Generation 6 - Current best internal CV score: -0.010626407922470287

Generation 7 - Current best internal CV score: -0.010626407922470287

Best pipeline: XGBRegressor(RidgeCV(input_matrix), learning_rate=0.1, max_depth=3, min_child_weight=15, n_estimators=100, n_jobs=1, objective=reg:squarederror, subsample=0.7000000000000001, verbosity=0)


TPOTRegressor(config_dict={'sklearn.ensemble.GradientBoostingRegressor': {'alpha': [0.75,
                                                                                    0.8,
                                                                                    0.85,
                                                                                    0.9,
                                                                                    0.95,
                                                                                    0.99],
                                                                          'learning_rate': [0.001,
                                                                                            0.01,
                                                                                            0.1,
                                                                                            0.5,
                                                                      

In [36]:
preds = tpot.predict(test_pt)

print("\nBest pipeline steps:")
for i, (name, step) in enumerate(tpot.fitted_pipeline_.steps, start=1):
    print(f"{i}. {step}")


Best pipeline steps:
1. StackingEstimator(estimator=RidgeCV(alphas=array([ 0.1,  1. , 10. ])))
2. XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, gamma=0, gpu_id=-1,
             importance_type='gain', interaction_constraints='',
             learning_rate=0.1, max_delta_step=0, max_depth=3,
             min_child_weight=15, missing=nan, monotone_constraints='()',
             n_estimators=100, n_jobs=1, num_parallel_tree=1, random_state=42,
             reg_alpha=0, reg_lambda=1, scale_pos_weight=1,
             subsample=0.7000000000000001, tree_method='exact',
             validate_parameters=1, verbosity=0)


In [37]:
preds

array([0.40438768, 0.5397139 , 0.5276115 , ..., 0.5781705 , 0.45213082,
       0.5315423 ], dtype=float32)

## Submit your output csv

In [38]:
submission=pd.read_csv("/kaggle/input/zelestra/test.csv")

In [39]:
submission_df =pd.DataFrame({'id': submission['id'].values, 'efficiency': preds})

In [40]:
submission_df.to_csv("highscore.csv",index=False)